In [1]:
pip install -q faiss-cpu sentence-transformers rank-bm25 bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 86.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 90.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 86.6 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Note: you may need to restart the kernel to use updated packages.


In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

print("Hugging Face login successful")

Hugging Face login successful


In [ ]:
## fusion dia merge (submssion 4, 6, 7 e eta use kora hoise)

import faiss
import pickle
import torch
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

# ----------------------------------------------------------
# PATHS
# ----------------------------------------------------------
TEST_PATH = "/kaggle/input/competitions/are-you-sure-llm-is-enough-intra-cuet-ml-contest-2-0/Dataset/test.csv"



CHUNKS_PATH = "/kaggle/input/models/nirjharami/sub8-chunk/other/default/1/chunks_recommended2.pkl" ### sub 8 er ta dia try kora jai
EMBED_PATH = "/kaggle/input/models/nirjharami/sub8-embed/other/default/1/bge_m3_embeddings_recommended2.npy"

# ----------------------------------------------------------
# LOAD CHUNKS
# ----------------------------------------------------------
print("Loading chunks...")
with open(CHUNKS_PATH, "rb") as f:
    chunks = pickle.load(f)
print("Chunks loaded:", len(chunks))

# ----------------------------------------------------------
# LOAD EMBEDDINGS
# ----------------------------------------------------------
print("Loading BGE-M3 embeddings...")
chunk_embeddings = np.load(EMBED_PATH)
print("Embeddings shape:", chunk_embeddings.shape)

# ----------------------------------------------------------
# BM25
# ----------------------------------------------------------
print("Building BM25...")
tokenized_corpus = [c.split() for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

# ----------------------------------------------------------
# LOAD BGE-M3
# ----------------------------------------------------------
print("Loading BGE-M3 model...")
embed_model = SentenceTransformer(
    "BAAI/bge-m3",
    device="cpu"
)
test_embed = embed_model.encode(
    ["test"],
    normalize_embeddings=True
)
print("Model dimension:", test_embed.shape[1])

# ----------------------------------------------------------
# FAISS
# ----------------------------------------------------------
print("Building FAISS index...")
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)
print("FAISS size:", index.ntotal)

# ----------------------------------------------------------
# LOAD RERANKER
# ----------------------------------------------------------
print("Loading reranker...")
reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3",
    device="cuda"
)

# =========================================================
# IMPROVED RETRIEVAL FUNCTION WITH WEIGHTED MERGING
# =========================================================

def retrieve(question,
             bm25_k=5,
             dense_k=15,
             final_k=2,
             bm25_weight=0.7,
             dense_weight=0.3,
             verbose=True):
    """
    Improved retrieval with weighted merging
    + previous chunk augmentation.
    """

    # =====================================================
    # STEP 1: BM25 RETRIEVAL
    # =====================================================
    if verbose:
        print(f"[BM25] Retrieving top {bm25_k}...")

    bm25_scores = bm25.get_scores(question.split())
    bm25_idx = np.argsort(bm25_scores)[::-1][:bm25_k]
    bm25_scores_top = bm25_scores[bm25_idx]

    if verbose:
        print(f"  BM25 scores: {bm25_scores_top}")

    # =====================================================
    # STEP 2: DENSE RETRIEVAL
    # =====================================================
    if verbose:
        print(f"[Dense] Retrieving top {dense_k}...")

    q_embedding = embed_model.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    dense_scores, dense_idx = index.search(
        q_embedding,
        dense_k
    )

    dense_idx = dense_idx[0]
    dense_scores = dense_scores[0]

    if verbose:
        print(f"  Dense scores: {dense_scores}")

    # =====================================================
    # STEP 3: WEIGHTED MERGE
    # =====================================================
    if verbose:
        print(
            f"[Merge] Merging with weights "
            f"(BM25={bm25_weight}, Dense={dense_weight})..."
        )

    # Normalize BM25 scores
    if len(bm25_scores_top) > 1:
        bm25_min = bm25_scores_top.min()
        bm25_max = bm25_scores_top.max()

        bm25_norm = (
            (bm25_scores_top - bm25_min)
            / (bm25_max - bm25_min + 1e-8)
        )
    else:
        bm25_norm = np.array([1.0])

    # Normalize Dense scores
    if len(dense_scores) > 1:
        dense_min = dense_scores.min()
        dense_max = dense_scores.max()

        dense_norm = (
            (dense_scores - dense_min)
            / (dense_max - dense_min + 1e-8)
        )
    else:
        dense_norm = np.array([1.0])

    # =====================================================
    # STEP 4: SCORE MERGING
    # =====================================================
    scores_dict = {}

    # BM25
    for idx, score in zip(bm25_idx, bm25_norm):
        scores_dict[int(idx)] = (
            bm25_weight * float(score)
        )

    # Dense
    for idx, score in zip(dense_idx, dense_norm):

        idx_int = int(idx)

        if idx_int in scores_dict:
            scores_dict[idx_int] += (
                dense_weight * float(score)
            )
        else:
            scores_dict[idx_int] = (
                dense_weight * float(score)
            )

    # Sort candidates
    sorted_candidates = sorted(
        scores_dict.items(),
        key=lambda x: x[1],
        reverse=True
    )

    merge_k = max(
        final_k * 2,
        len(sorted_candidates)
    )

    candidate_ids = [
        idx
        for idx, _
        in sorted_candidates[:merge_k]
    ]

    if verbose:
        print(
            f"  Merged {len(candidate_ids)} "
            f"candidates (before reranking)"
        )

        print(
            f"  Merged scores: "
            f"{sorted_candidates[:3]}"
        )

    # =====================================================
    # STEP 5: PREPARE FOR RERANKING
    # =====================================================
    candidate_chunks = [
        (idx, chunks[idx])
        for idx in candidate_ids
    ]

    # =====================================================
    # STEP 6: RERANK
    # =====================================================
    if verbose:
        print(
            f"[Rerank] Reranking "
            f"{len(candidate_chunks)} candidates..."
        )

    pairs = [
        (question, chunk_text)
        for _, chunk_text
        in candidate_chunks
    ]

    rerank_scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidate_chunks, rerank_scores),
        key=lambda x: x[1],
        reverse=True
    )

    # =====================================================
    # STEP 7: ADD PREVIOUS CHUNK
    # =====================================================
    best_chunks = []
    best_scores = []

    seen = set()

    for ((idx, chunk_text), score) in ranked[:final_k]:

        merged_parts = []

        # Add previous chunk only
        if idx > 0 and (idx - 1) not in seen:
            merged_parts.append(
                chunks[idx - 1]
            )
            seen.add(idx - 1)

        # Add current chunk
        if idx not in seen:
            merged_parts.append(
                chunk_text
            )
            seen.add(idx)

        merged_chunk = "\n".join(
            merged_parts
        )

        best_chunks.append(
            merged_chunk
        )

        best_scores.append(score)

    # =====================================================
    # STEP 8: FINAL OUTPUT
    # =====================================================
    confidence = np.mean(best_scores)
    confidence = float(
        np.clip(confidence, 0.0, 1.0)
    )

    context = "\n\n".join(
        best_chunks
    )

    if verbose:
        print(
            f"[Result] Selected top "
            f"{final_k} chunks"
        )

        print(
            f"  Rerank scores: "
            f"{best_scores}"
        )

        print(
            f"  Confidence: "
            f"{confidence:.2%}\n"
        )

    return context, confidence


# =========================================================
# ALTERNATIVE: WEIGHTED MERGE WITH RATIO (Simpler)
# =========================================================

def retrieve_v2(question,
                bm25_k=5,
                dense_k=5,
                final_k=3):
    """
    Simpler version using rank-based weighting.
    """
    
    # BM25 retrieval
    bm25_scores = bm25.get_scores(question.split())
    bm25_idx = np.argsort(bm25_scores)[::-1][:bm25_k]
    
    # Dense retrieval
    q_embedding = embed_model.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    scores, indices = index.search(q_embedding, dense_k)
    dense_idx = indices[0]
    
    # WEIGHTED MERGE using rank position
    scores_dict = {}
    
    # BM25: earlier rank = higher weight
    for rank, idx in enumerate(bm25_idx):
        scores_dict[int(idx)] = 0.4 * (1.0 - rank / bm25_k)
    
    # Dense: earlier rank = higher weight
    for rank, idx in enumerate(dense_idx):
        idx_int = int(idx)
        dense_score = 0.6 * (1.0 - rank / dense_k)
        if idx_int in scores_dict:
            scores_dict[idx_int] += dense_score
        else:
            scores_dict[idx_int] = dense_score
    
    # Sort and get top candidates
    sorted_candidates = sorted(
        scores_dict.items(),
        key=lambda x: x[1],
        reverse=True
    )
    
    candidate_ids = [idx for idx, _ in sorted_candidates[:final_k*2]]
    candidate_chunks = [chunks[i] for i in candidate_ids]
    
    # RERANK
    pairs = [(question, chunk) for chunk in candidate_chunks]
    rerank_scores = reranker.predict(pairs)
    
    ranked = sorted(
        zip(candidate_chunks, rerank_scores),
        key=lambda x: x[1],
        reverse=True
    )
    
    best_chunks = [x[0] for x in ranked[:final_k]]
    context = "\n\n".join(best_chunks)
    confidence = np.mean([x[1] for x in ranked[:final_k]])
    
    return context, confidence


# =========================================================
# USAGE EXAMPLES
# =========================================================

if __name__ == "__main__":
    
    # Example question
    question = "ম্যাক্স ব্যারনের বন্ধুরা তাকে কেন প্রায়ই ঠাট্টা-বিদ্রুপ করে?"
    
    print("=" * 70)
    print("RETRIEVAL WITH WEIGHTED MERGING")
    print("=" * 70)
    
    # Method 1: Score-based weighted merging
    print("\n📌 METHOD 1: Score-based Weighted Merging")
    print("-" * 70)
    context, confidence = retrieve(
        question,
        bm25_k=5,
        dense_k=5,
        final_k=3,
        bm25_weight=0.4,
        dense_weight=0.6
    )
    
    print(f"Context:\n{context}\n")
    print(f"Confidence: {confidence:.2%}\n")
    
    # Method 2: Rank-based weighted merging (simpler)
    print("\n📌 METHOD 2: Rank-based Weighted Merging")
    print("-" * 70)
    context2, confidence2 = retrieve_v2(
        question,
        bm25_k=5,
        dense_k=5,
        final_k=3
    )
    
    print(f"Context:\n{context2}\n")
    print(f"Confidence: {confidence2:.2%}\n")
    
    # Compare which is better
    print("\n" + "=" * 70)
    print("COMPARISON")
    print("=" * 70)
    print(f"Method 1 confidence: {confidence:.2%}")
    print(f"Method 2 confidence: {confidence2:.2%}")
    print(f"Difference: {abs(confidence - confidence2):.2%}")

Loading chunks...
Chunks loaded: 75819
Loading BGE-M3 embeddings...
Embeddings shape: (75819, 1024)
Building BM25...
Loading BGE-M3 model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

RETRIEVAL WITH WEIGHTED MERGING

📌 METHOD 1: Score-based Weighted Merging
----------------------------------------------------------------------
[BM25] Retrieving top 5...
  BM25 scores: [42.86846087 23.64831736 19.11533572 15.50432351 14.50538848]
[Dense] Retrieving top 5...
  Dense scores: [0.57791597 0.5670579  0.5227811  0.5053871  0.5023142 ]
[Merge] Merging with weights (BM25=0.4, Dense=0.6)...
  Merged 8 candidates (before reranking)
  Merged scores: [(25778, 0.9999999283333978), (25780, 0.527914708298363), (29256, 0.16243164539337157)]
[Rerank] Reranking 8 candidates...
[Result] Selected top 3 chunks
  Rerank scores: [np.float32(0.9944946), np.float32(0.14005448), np.float32(0.054799486)]
  Confidence: 39.64%

Context:
সেন্ট লুইস, মিসৌরিতে
একজন মধ্যবয়সী শ্রমিক শ্রেণীর
খাদ্য পরিবেশিকা
(সার‍্যান্ডন)-এর প্রেমে পড়েন। ছবিটির মূল সঙ্গীত স্কোর জর্জ ফেন্টন দ্বারা রচিত হয়েছিল। "একজন তরুণ পুরুষ এবং একজন সাহসী মহিলার গল্প" ("The story of a younger man and a bolder woman") ট্যাগলাইন দিয

In [5]:
## submission 8 settings দিয়ে test retrieval save

import pandas as pd
from tqdm.auto import tqdm
import gc
import torch

# =====================================================
# LOAD TEST CSV
# =====================================================

TEST_PATH = "/kaggle/input/competitions/are-you-sure-llm-is-enough-intra-cuet-ml-contest-2-0/Dataset/test.csv"

df = pd.read_csv(TEST_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# =====================================================
# STORAGE
# =====================================================

context1_list = []
context2_list = []

confidence1_list = []
confidence2_list = []

# =====================================================
# GENERATE RETRIEVALS
# =====================================================

tqdm_bar = tqdm(
    range(len(df)),
    desc="Generating test contexts"
)

for idx in tqdm_bar:

    try:

        question = str(
            df.iloc[idx]["question"]
        ).strip()

        # =================================================
        # CONTEXT 1
        # submission8 first retrieval
        # =================================================

        context1, confidence1 = retrieve(
            question,
            bm25_k=10,
            dense_k=15,
            final_k=6,
            bm25_weight=0.65,
            dense_weight=0.35,
            verbose=False
        )

        # =================================================
        # CONTEXT 2
        # submission8 retry retrieval
        # =================================================

        context2, confidence2 = retrieve(
            question,
            bm25_k=10,
            dense_k=15,
            final_k=10,
            bm25_weight=0.65,
            dense_weight=0.35,
            verbose=False
        )

        # =================================================
        # STORE
        # =================================================

        context1_list.append(context1)
        context2_list.append(context2)

        confidence1_list.append(
            confidence1
        )

        confidence2_list.append(
            confidence2
        )

        # =================================================
        # SHOW FIRST 3
        # =================================================

        if idx < 3:

            print("\n" + "=" * 100)
            print(f"QUESTION {idx+1}")
            print("=" * 100)

            print("\nQUESTION:")
            print(question)

            print("\nCONTEXT 1:")
            print(context1[:500])

            print("\nCONFIDENCE 1:")
            print(confidence1)

            print("\nCONTEXT 2:")
            print(context2[:500])

            print("\nCONFIDENCE 2:")
            print(confidence2)

        # =================================================
        # TEMP SAVE EVERY 100
        # =================================================

        if (idx + 1) % 100 == 0:

            temp_df = df.iloc[
                :len(context1_list)
            ].copy()

            temp_df["context1"] = (
                context1_list
            )

            temp_df["context2"] = (
                context2_list
            )

            temp_df["confidence1"] = (
                confidence1_list
            )

            temp_df["confidence2"] = (
                confidence2_list
            )

            temp_df.to_csv(
                "test_with_rag_temp.csv",
                index=False
            )

            print(
                f"\n✅ Saved progress:"
                f" {idx+1}/{len(df)}"
            )

        # =================================================
        # MEMORY CLEANUP
        # =================================================

        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:

        print(
            f"\n❌ Error at row {idx}"
        )
        print(e)

        context1_list.append("")
        context2_list.append("")

        confidence1_list.append(0.0)
        confidence2_list.append(0.0)

# =====================================================
# FINAL SAVE
# =====================================================

df["context1"] = context1_list
df["context2"] = context2_list

df["confidence1"] = confidence1_list
df["confidence2"] = confidence2_list

SAVE_NAME = "test_with_rag.csv"

df.to_csv(
    SAVE_NAME,
    index=False
)

print("\n✅ DONE")
print(f"Saved as: {SAVE_NAME}")

print("\nFinal columns:")
print(df.columns.tolist())

print("\nSample:")
print(
    df[
        [
            "index",
            "question",
            "context1",
            "context2"
        ]
    ].head(3)
)

Shape: (1500, 2)
Columns: ['index', 'question']


Generating test contexts:   0%|          | 0/1500 [00:00<?, ?it/s]


QUESTION 1

QUESTION:
অমলক রতন কোহলি কোন ক্ষেত্রে বিশিষ্ট হিসেবে সম্মানিত?

CONTEXT 1:
১৯১৬-এ জন্ম
১৯৯৪-এ মৃত্যু
পশ্চিমবঙ্গের প্রকৌশলী
২০শ শতাব্দীর ভারতীয় উদ্ভাবক
২০শ শতাব্দীর ভারতীয় প্রকৌশলী
ভারতীয় পেটেন্ট ধারক
কলকাতা বিশ্ববিদ্যালয়ের প্রাক্তন শিক্ষার্থী
ইম্পেরিয়াল কলেজ লন্ডনের প্রাক্তন শিক্ষার্থী
লুকানো বিষয়শ্রেণী:
এইচকার্ডের সাথে নিবন্ধসমূহ
এ পৃষ্ঠায় শেষ পরিবর্তন হয়েছিল ১৫:০৮টার সময়, ২৩ আগস্ট ২০২৫ তারিখে।
ক্ষিতীশরঞ্জন চক্রবর্তী
৪টি ভাষা
আলোচনা যোগ করুন
অমলক রতন কোহলি
অমলক রতন কোহলি - উইকিপিডিয়া
সূচনা
১
তথ্যসূত্র
২
বহিঃসংযোগ
সূচিপত্র টগল করুন
অমলক রতন কোহলি
৩টি ভাষা
Eng

CONFIDENCE 1:
0.40822336077690125

CONTEXT 2:
১৯১৬-এ জন্ম
১৯৯৪-এ মৃত্যু
পশ্চিমবঙ্গের প্রকৌশলী
২০শ শতাব্দীর ভারতীয় উদ্ভাবক
২০শ শতাব্দীর ভারতীয় প্রকৌশলী
ভারতীয় পেটেন্ট ধারক
কলকাতা বিশ্ববিদ্যালয়ের প্রাক্তন শিক্ষার্থী
ইম্পেরিয়াল কলেজ লন্ডনের প্রাক্তন শিক্ষার্থী
লুকানো বিষয়শ্রেণী:
এইচকার্ডের সাথে নিবন্ধসমূহ
এ পৃষ্ঠায় শেষ পরিবর্তন হয়েছিল ১৫:০৮টার সময়, ২৩ আগস্ট ২০২৫ তারিখে।
ক্ষিতীশরঞ্জন চক্রবর্তী
৪টি ভাষা